## 計算資料

計算設定aoi內 :

- 地圖資料(碰撞區、合法區)
- 地圖特徵點 (E nodes/ T nodes, E rings/T rings)
- 地圖門戶，為地圖特徵點轉變 (T gates)
- 海節點、海線邊 (Sea nodes/Sea Edges)
- 海與陸連結 T_S connectors
- E <-> T connectors
- 做成Cache
  - G_global,pkl.gz
  - out_global.pkl.gz

In [ ]:
from pathlib import Path
import routing_map
from routing_map import build_aoi, RoutingMapConfig
from routing_map.config import AoiConfig, LandConfig
from routing_map.ring_types import RingBuildConfig
from routing_map import cache_utils


cfg = RoutingMapConfig(
    aoi=AoiConfig(
        bbox_ll=((-179.999, -85, 179.999, 85)),
    ),
    land=LandConfig(
        shp_path=Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"),
        buffer_km=20.0,
        avoid_km= 1,
        collision_safety_km=0.5,
    ),
        rings=RingBuildConfig(
        clearance_m=10000.0,      # 1 km：你 land.avoid_km 的概念
        ring_sample_km=5.0,      # 采樣間距（先保守）
        taut_window_size=80,
        taut_max_tries=8,
        min_ring_length_km=20.0,

        taut_use_clearance_buffer= True,
        taut_collision_buffer_m= 5000,
    ),
    
)


CACHE_DIR = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\aoi_cache"
CACHE_TAG = "global"  # 你可以改成 "taiwan" / "world" / "dateline" 都行

out = cache_utils.get_out(cfg, cache_dir=CACHE_DIR, use_cache=True)



## 主程式

### 單條航線+畫圖debug (須建立out)

- 含hard_cap = 70，作為東北航線的開關

In [ ]:
from routing_map.pipeline import (
    run_p2p_multiworld,
    RoutePolicy,
    run_p2p,
    GraphConfig, SnapConfig, SimplifyConfig, RunConfig,
)
from routing_map.repairer import RepairConfig
from routing_map.viz_layers import (
    make_base_map,
    add_sea_layers, add_ring_layers, add_connector_layers,
    add_points_layer, add_path_layer,
    finalize_map,
)

# 1) 你 build_aoi(cfg) 得到 out 後：
bbox_ll = out["bbox_ll"]
print("bbox_ll:", bbox_ll)

# 2) 設定起終點（自己改）
origin_ll = (-126.26155, 45.69071)  
dest_ll   = (-55.48436, 18.73282)


# 3) 設定流程 config（你先用預設即可，之後再調參）
graph_cfg = GraphConfig(
    bbox_ll=bbox_ll,
    max_sea_edges=None,
    max_ring_edges=None,
    weight_unit="km",
)

snap_cfg = SnapConfig(
    k_near=30,
    r_max_km=150.0,
    k_inject=4,
)

repair_cfg = RepairConfig(
    debug=True,
)

simplify_cfg = SimplifyConfig(
    enabled=True,
    window_size=80,
    max_tries=300,
    use_prepared_collision=True,
    dateline_unwrap=True,
    wrap_output_lon=True,
)

run_cfg = RunConfig(
    do_repair=True,
    do_simplify=True,
    debug=True,
)

# ---- Policy: hard cap at |lat| <= 70 deg ----
policy = RoutePolicy(
    hard_lat_cap_deg=70.0,
    # 下面這些如果你 patched 版有提供，就照你想要的開關調：
    # enable_ne=False,
    # enable_nw=False,
    # enable_gateways=True,
)
# 4) 跑新 pipeline（內含：建圖 + snap/inject + A* + repair + simplify）

graph_build_args = dict(
    include_sea=graph_cfg.include_sea,
    include_cc=graph_cfg.include_cc,
    include_gateb_sea=graph_cfg.include_gateb_sea,
    include_c_gateb=graph_cfg.include_c_gateb,
    include_rings=graph_cfg.include_rings,
    include_et=graph_cfg.include_et,
    include_tgate_sea=graph_cfg.include_tgate_sea,
    max_sea_edges=graph_cfg.max_sea_edges,
    max_ring_edges=graph_cfg.max_ring_edges,
    weight_unit=graph_cfg.weight_unit,
    bbox_ll=bbox_ll,
)

G_base = cache_utils.get_graph(
    out,
    cfg=cfg,
    graph_build_args=graph_build_args,
    cache_dir=CACHE_DIR,
    use_cache=True,
)


res = run_p2p_multiworld(
    out,
    origin_ll,
    dest_ll,
    graph_cfg=graph_cfg,
    snap_cfg=SnapConfig(R_NEAR_COAST_KM=120.0, S_MAX_SNAP_KM=200.0),
    repair_cfg=repair_cfg,
    simplify_cfg=simplify_cfg,
    run_cfg=RunConfig(debug=True),
    G_in=G_base,
    policy=policy,
)

print("error:", res.error)
print("final points:", 0 if not res.path_ll_final else len(res.path_ll_final))
print("lengths_km:", res.lengths_km)

# 5) 畫圖（新版 layers）
m = make_base_map(bbox_ll=bbox_ll, zoom_start=5)

add_sea_layers(
    m, out,
    node_sample=20000,
    max_edges=20000,
    show=False,
    bbox_ll=bbox_ll,
)
add_ring_layers(
    m, out,
    e_node_sample=60000,
    t_node_sample=50000,
    max_e_edges=50000,
    max_t_edges=50000,
    show=False,
    bbox_ll=bbox_ll,
)
add_connector_layers(
    m, out,
    max_et=10000,
    max_tgate=5000,
    show=False,
    bbox_ll=bbox_ll,
)

# points
# add_points_layer(m, [origin_ll], name="origin_raw", radius=7, show=True, bbox_ll=bbox_ll)
# add_points_layer(m, [dest_ll],   name="dest_raw",   radius=7, show=True, bbox_ll=bbox_ll)
#if res.start_ll_snap:
    #add_points_layer(m, [res.start_ll_snap], name="origin_snap", radius=6, show=True, bbox_ll=bbox_ll)
#if res.end_ll_snap:
    #add_points_layer(m, [res.end_ll_snap],   name="dest_snap",   radius=6, show=True, bbox_ll=bbox_ll)

# paths（你想看哪條就開哪條）
if res.path_ll_raw:
    add_path_layer(m, res.path_ll_raw, name="path_raw", weight=3, opacity=0.6, show=False)
if res.path_ll_repaired:
    add_path_layer(m, res.path_ll_repaired, name="path_repaired", weight=4, opacity=0.7, show=False)
if res.path_ll_simplified:
    #add_path_layer(m, res.path_ll_simplified, name="path_simplified", weight=5, opacity=0.9, show=True)
    add_path_layer(
        m, res.path_ll_simplified,
        name="path_simplified",
        weight=5, opacity=0.9, show=True,
        geodesic=True,
        geodesic_step_km=20.0,   # 想更平滑就 10；想更快就 30~50
    )
elif res.path_ll_final:
    add_path_layer(m, res.path_ll_final, name="path_ll_final", weight=5, opacity=0.9, show=True)

# points
add_points_layer(m, [origin_ll], name="origin_raw", radius=7, show=True, bbox_ll=bbox_ll)
add_points_layer(m, [dest_ll],   name="dest_raw",   radius=7, show=True, bbox_ll=bbox_ll)
if res.start_ll_snap:
    add_points_layer(m, [res.start_ll_snap], name="origin_snap", radius=6, show=True, bbox_ll=bbox_ll)
if res.end_ll_snap:
    add_points_layer(m, [res.end_ll_snap],   name="dest_snap",   radius=6, show=True, bbox_ll=bbox_ll)

html = finalize_map(m, html_path="aoi_rings_p2p_debug.html")
from webbrowser import open as wb_open
wb_open("aoi_rings_p2p_debug.html")
print("saved:", html)


### 單條航線 只畫路線 (須建立out後)


In [ ]:
# ===== Batch runner: load out_global + G_global + hydrate (proj + sea_kdt) + run routes =====
import os, json, time, traceback, gzip, pickle
from datetime import datetime
from contextlib import redirect_stdout

import numpy as np
from scipy.spatial import cKDTree

from routing_map.pipeline import (
    run_p2p_multiworld,
    RoutePolicy,
    GraphConfig, SnapConfig, SimplifyConfig, RunConfig,
)
from routing_map.repairer import RepairConfig
from routing_map.geom_utils import build_projector_from_bbox

from routing_map.viz_layers import (
    make_base_map,
    add_path_layer,
    add_points_layer,
    finalize_map,
)

# =========================
# 0) Paths
# =========================
CACHE_DIR = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\aoi_cache"
OUT_PATH  = os.path.join(CACHE_DIR, "out_global.pkl.gz")
G_PATH    = os.path.join(CACHE_DIR, "G_global.pkl.gz")

def load_gz_pickle(path: str):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)

def safe_name(s: str) -> str:
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|']
    for ch in bad:
        s = s.replace(ch, "_")
    return s.replace(" ", "_")

def hydrate_out_for_pipeline(out: dict) -> dict:
    # ---- proj ----
    if out.get("proj") is None:
        bbox_ll = out.get("bbox_ll")
        if bbox_ll is None:
            raise KeyError("out['bbox_ll'] missing; cannot build projector")
        out["proj"] = build_projector_from_bbox(bbox_ll)

    # ---- collision_m (如果你的 pipeline/repair/simplify 會用) ----
    if out.get("collision_m") is None:
        if out.get("union_smooth_m") is not None:
            out["collision_m"] = out["union_smooth_m"]
        elif out.get("ring_base_m") is not None:
            out["collision_m"] = out["ring_base_m"]

    # ---- sea_kdt (snap.py 需要 out['S_nodes'] + out['sea_kdt']) ----
    if out.get("S_nodes") is None:
        raise KeyError("out['S_nodes'] missing (needed by snap.py)")

    if out.get("sea_kdt") is None:
        if out.get("S_xy_m") is not None:
            xy = np.asarray(out["S_xy_m"], dtype=float)
        else:
            S = out["S_nodes"]
            xy = S[["x_m", "y_m"]].to_numpy(dtype=float)
        out["sea_kdt"] = cKDTree(xy)
        out["sea_kdt_kind"] = "ckdtree"

    return out

# =========================
# 2) Load once
# =========================
out = load_gz_pickle(OUT_PATH)
G_pack = load_gz_pickle(G_PATH)
G_base = G_pack["G_base"] if isinstance(G_pack, dict) and "G_base" in G_pack else G_pack

print("[load] out keys:", len(out))
print("[load] G_base:", type(G_base), "nodes/edges:", G_base.number_of_nodes(), G_base.number_of_edges())

# =========================
# 3) Hydrate once
# =========================
out = hydrate_out_for_pipeline(out)
print("[hydrate] proj:", type(out["proj"]))
print("[hydrate] sea_kdt:", type(out["sea_kdt"]), "| S_nodes len:", len(out["S_nodes"]))
print("[hydrate] collision_m:", out.get("collision_m") is not None)

bbox_ll = out["bbox_ll"]

origin_ll = (100.9158073, 13.1040552)
dest_ll   = (114.39077,22.16387)

graph_cfg = GraphConfig(
    bbox_ll=bbox_ll,
    max_sea_edges=None,
    max_ring_edges=None,
    weight_unit="km",
)

snap_cfg = SnapConfig(
    k_near=30,
    r_max_km=150.0,
    k_inject=4,
    R_NEAR_COAST_KM=120.0,
    S_MAX_SNAP_KM=200.0,
)

repair_cfg = RepairConfig(debug=True)

simplify_cfg = SimplifyConfig(
    enabled=True,
    window_size=80,
    max_tries=300,
    use_prepared_collision=True,
    dateline_unwrap=True,
    wrap_output_lon=True,
)

run_cfg = RunConfig(
    do_repair=True,
    do_simplify=True,
    debug=True,
)

policy = RoutePolicy(hard_lat_cap_deg=70.0)

#  直接用你 load 出來的 G_base，不要再 get_graph
res = run_p2p_multiworld(
    out,
    origin_ll,
    dest_ll,
    graph_cfg=graph_cfg,
    snap_cfg=snap_cfg,
    repair_cfg=repair_cfg,
    simplify_cfg=simplify_cfg,
    run_cfg=run_cfg,
    G_in=G_base,
    policy=policy,
)

print("error:", res.error)
print("final points:", 0 if not res.path_ll_final else len(res.path_ll_final))
print("lengths_km:", res.lengths_km)

# --- draw ---
m = make_base_map(bbox_ll=bbox_ll, zoom_start=5)

if res.path_ll_raw:
    add_path_layer(m, res.path_ll_raw, name="path_raw", weight=3, opacity=0.6, show=False)
if res.path_ll_repaired:
    add_path_layer(m, res.path_ll_repaired, name="path_repaired", weight=4, opacity=0.7, show=False)
if res.path_ll_simplified:
    add_path_layer(
        m, res.path_ll_simplified,
        name="path_simplified",
        weight=5, opacity=0.9, show=True,
        geodesic=True,
        geodesic_step_km=20.0,
    )
elif res.path_ll_final:
    add_path_layer(m, res.path_ll_final, name="path_ll_final", weight=5, opacity=0.9, show=True)

add_points_layer(m, [origin_ll], name="origin_raw", radius=7, show=True, bbox_ll=bbox_ll)
add_points_layer(m, [dest_ll],   name="dest_raw",   radius=7, show=True, bbox_ll=bbox_ll)
if res.start_ll_snap:
    add_points_layer(m, [res.start_ll_snap], name="origin_snap", radius=6, show=True, bbox_ll=bbox_ll)
if res.end_ll_snap:
    add_points_layer(m, [res.end_ll_snap],   name="dest_snap",   radius=6, show=True, bbox_ll=bbox_ll)

html = finalize_map(m, html_path="aoi_rings_p2p_debug_onlypath.html")
import webbrowser
webbrowser.open("aoi_rings_p2p_debug_onlypath.html")
print("saved:", html)


### 不畫圖，僅存log，支援多航線，支援cache

 - 每次讀cache後須重建 sea KDTree


In [ ]:
# ===== Batch runner: load out_global + G_global + hydrate (proj + sea_kdt) + run routes =====
import os, json, time, traceback, gzip, pickle
from datetime import datetime
from contextlib import redirect_stdout

import numpy as np
from scipy.spatial import cKDTree

from routing_map.pipeline import (
    run_p2p_multiworld,
    RoutePolicy,
    GraphConfig, SnapConfig, SimplifyConfig, RunConfig,
)
from routing_map.repairer import RepairConfig
from routing_map.geom_utils import build_projector_from_bbox

# =========================
# 0) Paths
# =========================
CACHE_DIR = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\aoi_cache"
OUT_PATH  = os.path.join(CACHE_DIR, "out_global.pkl.gz")
G_PATH    = os.path.join(CACHE_DIR, "G_global.pkl.gz")

# =========================
# 1) Routes
# =========================
ROUTES = [
    ("婆羅洲繞島測試", (114.12681, -3.47298), (110.26443, 2.24094)),
    ("澳洲繞島", (145.95915, -16.89399), (115.24954, -30.53309)),
    ("斯里蘭卡繞島", (50.053, -7.580), (45.406, -30.448)),
    ("斯里蘭卡繞島2", (48.076, -7.188), (45.0, -27.683)),
    ("大連到韓國東邊", (129.1462, 37.6763), (118.94385, 38.21293)),
    ("曼谷到海上", (102.67236, 11.72813), (63.14977, 23.74978)),
    ("印尼東邊島嶼到印尼西邊島", (144.085, -3.827), (100.546, -4.39)),
    ("菲律賓到澳洲北", (117.1646, -20.78385), (120.70506, 12.99051)),
    ("高雄到非洲南端", (120.3181, 22.58425), (39.04207, -17.05363)),
    ("高雄到澳洲南端", (120.3181, 22.58425), (129.31673, -31.70924)),
    ("日本到南非", (134.121, 35.889), (13.886, -26.588)),
    ("上海到義大利", (121.499, 31.365), (15.17, 36.683)),
    ("測試跨日線用", (119.5, -15.928), (-176.0, -33.0)),
    ("新加坡 -> LA", (95.34036, 3.92524), (-126.26155, 45.69071)),
    ("波特蘭到歐洲", (-126.26155, 45.69071), (-9.25365, 45.47495)),
    ("巴拿馬測試aoi", (-126.26155, 45.69071), (-55.48436, 18.73282)),
    ("紐約到新加坡", (-73.89133, 37.75587), (105.17552, 1.63965)),
    ("澳洲東邊至歐洲_A", (165.937, -17.978), (-11.25, 54.775)),
    ("澳洲東邊至歐洲_B", (158.90859, -17.4813), (-11.25, 54.775)),
    ("澳洲東邊至南美洲東邊", (-55.85763, -43.9542), (155.566, -17.978)),
    ("台中到雪梨", (120.508, 24.181), (151.617, -32.989)),
]

def load_gz_pickle(path: str):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)

def safe_name(s: str) -> str:
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|']
    for ch in bad:
        s = s.replace(ch, "_")
    return s.replace(" ", "_")

def hydrate_out_for_pipeline(out: dict) -> dict:
    # ---- proj ----
    if out.get("proj") is None:
        bbox_ll = out.get("bbox_ll")
        if bbox_ll is None:
            raise KeyError("out['bbox_ll'] missing; cannot build projector")
        out["proj"] = build_projector_from_bbox(bbox_ll)

    # ---- collision_m (如果你的 pipeline/repair/simplify 會用) ----
    if out.get("collision_m") is None:
        if out.get("union_smooth_m") is not None:
            out["collision_m"] = out["union_smooth_m"]
        elif out.get("ring_base_m") is not None:
            out["collision_m"] = out["ring_base_m"]

    # ---- sea_kdt (snap.py 需要 out['S_nodes'] + out['sea_kdt']) ----
    if out.get("S_nodes") is None:
        raise KeyError("out['S_nodes'] missing (needed by snap.py)")

    if out.get("sea_kdt") is None:
        if out.get("S_xy_m") is not None:
            xy = np.asarray(out["S_xy_m"], dtype=float)
        else:
            S = out["S_nodes"]
            xy = S[["x_m", "y_m"]].to_numpy(dtype=float)
        out["sea_kdt"] = cKDTree(xy)
        out["sea_kdt_kind"] = "ckdtree"

    return out

# =========================
# 2) Load once
# =========================
out = load_gz_pickle(OUT_PATH)
G_pack = load_gz_pickle(G_PATH)
G_base = G_pack["G_base"] if isinstance(G_pack, dict) and "G_base" in G_pack else G_pack

print("[load] out keys:", len(out))
print("[load] G_base:", type(G_base), "nodes/edges:", G_base.number_of_nodes(), G_base.number_of_edges())

# =========================
# 3) Hydrate once
# =========================
out = hydrate_out_for_pipeline(out)
print("[hydrate] proj:", type(out["proj"]))
print("[hydrate] sea_kdt:", type(out["sea_kdt"]), "| S_nodes len:", len(out["S_nodes"]))
print("[hydrate] collision_m:", out.get("collision_m") is not None)

# =========================
# 4) Configs
# =========================
bbox_ll = out["bbox_ll"]

graph_cfg = GraphConfig(bbox_ll=bbox_ll, max_sea_edges=None, max_ring_edges=None, weight_unit="km")
snap_cfg = SnapConfig(k_near=30, r_max_km=150.0, k_inject=4, R_NEAR_COAST_KM=120.0, S_MAX_SNAP_KM=200.0)
repair_cfg = RepairConfig(debug=True)
simplify_cfg = SimplifyConfig(
    enabled=True, window_size=80, max_tries=300,
    use_prepared_collision=True, dateline_unwrap=True, wrap_output_lon=True
)
run_cfg = RunConfig(do_repair=True, do_simplify=True, debug=True)
policy = RoutePolicy(hard_lat_cap_deg=70.0)

# =========================
# 5) Output dirs
# =========================
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join("batch_runs", ts)
LOG_DIR = os.path.join(RUN_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
summary_path = os.path.join(RUN_DIR, "summary.jsonl")

print("\nRUN_DIR =", RUN_DIR)
print("[batch] routes =", len(ROUTES))

# =========================
# 6) Run batch
# =========================
for i, (name, origin_ll, dest_ll) in enumerate(ROUTES, start=1):
    log_file = os.path.join(LOG_DIR, f"{i:02d}_{safe_name(name)}.log")
    t0 = time.time()
    res = None
    exc = None

    with open(log_file, "w", encoding="utf-8") as f:
        f.write(f"[batch] idx={i}/{len(ROUTES)} name={name}\n")
        f.write(f"[batch] origin_ll={origin_ll} dest_ll={dest_ll}\n")
        f.write(f"[batch] start_utc={datetime.utcnow().isoformat()}Z\n\n")
        f.flush()

        try:
            with redirect_stdout(f):
                res = run_p2p_multiworld(
                    out, origin_ll, dest_ll,
                    graph_cfg=graph_cfg,
                    snap_cfg=snap_cfg,
                    repair_cfg=repair_cfg,
                    simplify_cfg=simplify_cfg,
                    run_cfg=run_cfg,
                    G_in=G_base,
                    policy=policy,
                )
        except Exception as e:
            exc = f"{type(e).__name__}: {e}"
            f.write("\n\n[EXCEPTION]\n")
            f.write(traceback.format_exc())
            f.flush()

    dt = time.time() - t0

    error_str = getattr(res, "error", None) if res is not None else None
    final_km = None
    final_points = 0
    if res is not None:
        lengths_km = getattr(res, "lengths_km", None)
        if isinstance(lengths_km, dict):
            final_km = lengths_km.get("final")
        path_final = getattr(res, "path_ll_final", None)
        if path_final:
            final_points = len(path_final)

    row = dict(
        idx=i, name=name,
        origin_ll=origin_ll, dest_ll=dest_ll,
        elapsed_sec=round(dt, 3),
        exception=exc,
        error=error_str,
        final_km=final_km,
        final_points=final_points,
        log_file=os.path.relpath(log_file, RUN_DIR),
    )
    with open(summary_path, "a", encoding="utf-8") as sf:
        sf.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"[{i:02d}/{len(ROUTES)}] {name} | sec={dt:.1f} | error={error_str} | exc={exc}")

print("\nDONE.")
print(" -", RUN_DIR)
print(" -", summary_path)
print(" -", LOG_DIR)


### 單條航線 僅檢視LOG

In [ ]:
from routing_map.pipeline import run_p2p_multiworld, GraphConfig, SnapConfig, SimplifyConfig, RunConfig, RoutePolicy
from routing_map.repairer import RepairConfig

name, origin_ll, dest_ll = ("上海到義大利", (121.499, 31.365), (15.17, 36.683))

graph_cfg = GraphConfig(bbox_ll=out["bbox_ll"], weight_unit="km")
snap_cfg  = SnapConfig(k_near=30, r_max_km=150.0, k_inject=4, R_NEAR_COAST_KM=120.0, S_MAX_SNAP_KM=200.0)
repair_cfg = RepairConfig(debug=False)
simplify_cfg = SimplifyConfig(enabled=True, window_size=80, max_tries=300, use_prepared_collision=True, dateline_unwrap=True, wrap_output_lon=True)

#  只為了抓原因：debug 打開
run_cfg = RunConfig(do_repair=True, do_simplify=True, debug=True)
policy = RoutePolicy(hard_lat_cap_deg=70.0)

def dbg_out(out):
    keys = list(out.keys())
    print("[dbg] top keys sample:", keys[:20])
    for k in ["S_nodes","S_nodes_df","sea_nodes_df","sea_kdt","S_kdt","sea_kdtree","sea_kdt_kind"]:
        v = out.get(k, None)
        print(f"[dbg] {k}:",
              "MISSING" if v is None else type(v),
              "" if v is None else (f"len={len(v)}" if hasattr(v, "__len__") else ""))

dbg_out(out)


res = run_p2p_multiworld(
    out, origin_ll, dest_ll,
    graph_cfg=graph_cfg, snap_cfg=snap_cfg,
    repair_cfg=repair_cfg, simplify_cfg=simplify_cfg,
    run_cfg=run_cfg, G_in=G_base, policy=policy
)
print("error:", res.error)

